# KcBERT Fine-tuning on Shopping Reviews
Fine-tune **beomi/KcBERT-base** on `Fine_tuning_shopping.txt` (TSV: text\tlabel).
Outputs metrics and saves to `./kcbert_official_finetune/`. Edit the first cell to change paths.

In [1]:
# === 0) Config ===
from pathlib import Path
DATA_PATH = Path('Fine_tuning_shopping.txt')
MODEL_ID  = 'beomi/KcBERT-base'
OUT_DIR   = Path('./kcbert_official_finetune')
MAX_LEN   = 128
BATCH     = 32
EPOCHS    = 3
LR        = 3e-5
WARMUP    = 0.06
SEED      = 7
NUM_WORKERS = 2
print('DATA_PATH =', DATA_PATH.resolve()); print('MODEL_ID  =', MODEL_ID); print('OUT_DIR   =', OUT_DIR.resolve())

DATA_PATH = C:\workspace\multi02_data_science\project\real_kcbert\Fine_tuning_shopping.txt
MODEL_ID  = beomi/KcBERT-base
OUT_DIR   = C:\workspace\multi02_data_science\project\real_kcbert\kcbert_official_finetune


In [3]:
# === 2) Load & clean data ===
import pandas as pd, csv
assert DATA_PATH.exists(), f'File not found: {DATA_PATH}'
df = pd.read_csv(DATA_PATH, sep='\t', header=None, names=['text','label'], dtype={'text':str},
                 engine='python', quoting=csv.QUOTE_NONE, on_bad_lines='skip', encoding_errors='ignore')
df = df.dropna(subset=['text']).reset_index(drop=True)
df['text'] = (df['text'].str.replace(r"[\u200B-\u200D\uFEFF]", '', regex=True)
                         .str.replace(r"\s+", ' ', regex=True)
                         .str.strip())
df['label'] = pd.to_numeric(df['label'], errors='coerce').fillna(0).astype(int).clip(0,1)
df = df[df['text'].str.len() >= 3]
df = df.drop_duplicates(subset=['text','label']).reset_index(drop=True)
print(df.head(5)); print('Samples:', len(df)); print('Label dist:\n', df['label'].value_counts())

                                                text  label
0  시계가 약이 없는지 안가네요. 검수후 보내주셨음 좋았을거 같아요. 선물인데 시간이 ...      0
1                                   재질도 별로도 차에 맞지않네요      0
2                    완전 투명한줄 알았는데 노란빛깔이 띄어요 ㅠ너무 아쉬워요      1
3                                      저렴하니 그냥 쓸만합니다      1
4  좋아요 강쥐때문에 베란다입구막으려고 샀어요 애견펜스쳐놧다가 발에계속걸리고 넘어지고 ...      1
Samples: 190205
Label dist:
 label
1    117454
0     72751
Name: count, dtype: int64


In [4]:
# === 3) Split ===
from sklearn.model_selection import train_test_split
if df['label'].value_counts().min() < 2:
    train_df, valid_df = train_test_split(df, test_size=0.1, random_state=SEED)
else:
    train_df, valid_df = train_test_split(df, test_size=0.1, random_state=SEED, stratify=df['label'])
len(train_df), len(valid_df)

(171184, 19021)

In [5]:
# === 4) Tokenizer/Model ===
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer.model_max_length = MAX_LEN
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=2, problem_type='single_label_classification',
    id2label={0:'NEG',1:'POS'}, label2id={'NEG':0,'POS':1}
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at beomi/KcBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
# === 5) Datasets + tokenize ===
from datasets import Dataset
def tok_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)
train_tok = Dataset.from_pandas(train_df, preserve_index=False).map(tok_fn, batched=True, remove_columns=['text'])
valid_tok = Dataset.from_pandas(valid_df, preserve_index=False).map(tok_fn, batched=True, remove_columns=['text'])
train_tok, valid_tok

Map: 100%|██████████| 19021/19021 [00:01<00:00, 17964.72 examples/s]


(Dataset({
     features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 171184
 }),
 Dataset({
     features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 19021
 }))

In [7]:
# === 6) Trainer ===
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import numpy as np
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'precision': p, 'recall': r, 'f1': f1}
args = TrainingArguments(
    output_dir=str(OUT_DIR), overwrite_output_dir=True, do_train=True, do_eval=True,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
    learning_rate=LR, weight_decay=0.01, num_train_epochs=EPOCHS, warmup_ratio=WARMUP,
    logging_steps=100, save_steps=1000, save_total_limit=2,
    fp16=True, gradient_checkpointing=True, seed=SEED, dataloader_num_workers=NUM_WORKERS,
    report_to=[], save_safetensors=True,
)
trainer = Trainer(model=model, args=args, train_dataset=train_tok, eval_dataset=valid_tok,
                  data_collator=data_collator, tokenizer=tokenizer, compute_metrics=compute_metrics)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_26708\2007145377.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, args=args, train_dataset=train_tok, eval_dataset=valid_tok,


In [8]:
# === 7) Train & Evaluate ===
train_out = trainer.train()
eval_out  = trainer.evaluate()
eval_out

Step,Training Loss
100,0.688700
200,0.376600
300,0.317500
400,0.280800
500,0.278700
600,0.271200
700,0.273400
800,0.269600
900,0.244400
1000,0.257800


{'eval_loss': 0.3036395311355591,
 'eval_accuracy': 0.9341254403028232,
 'eval_precision': 0.954203099298762,
 'eval_recall': 0.938361995572961,
 'eval_f1': 0.9462162510194445,
 'eval_runtime': 40.0778,
 'eval_samples_per_second': 474.602,
 'eval_steps_per_second': 14.846,
 'epoch': 3.0}

In [9]:
# === 8) Confusion Matrix ===
pred = trainer.predict(valid_tok)
import numpy as np
from sklearn.metrics import confusion_matrix
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=-1)
confusion_matrix(y_true, y_pred, labels=[0,1])

array([[ 6746,   529],
       [  724, 11022]])

In [10]:
# === 9) Save ===
model.config.id2label = {0:'NEG',1:'POS'}
model.config.label2id = {'NEG':0,'POS':1}
trainer.save_model(str(OUT_DIR))
tokenizer.save_pretrained(str(OUT_DIR))
print('Saved to:', OUT_DIR.resolve())

Saved to: C:\workspace\multi02_data_science\project\real_kcbert\kcbert_official_finetune
